# 1kg_eur - rare PCA: loadings and PCs by MAF fraction, non-HM3, PC1-10

Non-HM3 arm only. Every panel is one of the four equal-count MAF fractions from
`rare_pca_fractionate.ipynb`, with the full arm as a reference column.

| plot | layout |
|---|---|
| Loadings | 10 rows (PC1-10) x one column per fraction, Manhattan by genomic position |
| PC scatters | one row per fraction x 5 columns (PC1-2, 3-4, 5-6, 7-8, 9-10), hexbin |
| Cross-fraction PC correlation | |r| between each fraction's PCs and the full arm's |

Loadings are drawn against **true genomic position**, not variant index, so the
width of an implicated interval is visible rather than just its existence. Each
row shares a y-axis across fractions, so a taller spike in one fraction really is
a bigger share of that PC - not a rescaled axis.

PC scatters are hexbin with log counts: at 223K individuals a scatter is a solid
blob, and the isolated low-count cells are exactly where family structure and
outliers live.

MAF ranges are read from each fit's own `.acount`, so the labels are the fitted
frequencies rather than the nominal bin edges.

## Config

In [ ]:
import os, glob, re, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

SAMPLE_SET = "1kg_eur"
WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
OUT_GS = f"{WS_GS}/{SAMPLE_SET}/01_ancestry/rare_pca_plots"

RARE = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_rare")
OUT  = f"{RARE}/plots"
os.makedirs(OUT, exist_ok=True)

ARM           = "nonhm3_rare_all"
N_PCS         = 10
INCLUDE_FULL  = True    # full-arm fit as a reference column

# fractions, in MAF order
frac_stems = sorted(glob.glob(f"{RARE}/fractionate/{ARM}_b*.eigenvec"),
                    key=lambda p: int(re.search(r"_b(\d+)\.eigenvec$", p).group(1)))
FITS = {}
if INCLUDE_FULL and os.path.isfile(f"{RARE}/unrel/{ARM}_all.eigenvec"):
    FITS["full arm"] = f"{RARE}/unrel/{ARM}_all"
for p in frac_stems:
    b = re.search(r"_b(\d+)\.eigenvec$", p).group(1)
    FITS[f"frac {b}"] = p[: -len(".eigenvec")]

assert FITS, f"no fits found for {ARM} - run rare_pca_fractionate.ipynb first"
for k, v in FITS.items():
    print(f"  {k:10s} {v}")

## Load, and label each fraction by its fitted MAF range

MAF comes from each fit's `.acount`, so the label reflects what actually went
into that PCA.

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS + 1)]

def read_eigenvec(stem):
    d = pd.read_csv(f"{stem}.eigenvec", sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: "person_id"})
    d["person_id"] = d["person_id"].astype(str)
    return d.set_index("person_id")[[c for c in PC_COLS if c in d.columns]]

def read_weights(stem):
    L = pd.read_csv(f"{stem}.eigenvec.allele", sep=r"\s+")
    idc = "#ID" if "#ID" in L.columns else "ID"
    parts = L[idc].str.split(":", n=2, expand=True)
    L = L.rename(columns={idc: "ID"})
    L["CHROM"] = parts[0].astype(str).str.replace("^chr", "", regex=True)
    L["POS"] = pd.to_numeric(parts[1], errors="coerce")
    L = L[L["CHROM"].str.fullmatch(r"\d+")].dropna(subset=["POS"])
    L["CHROM_N"] = L["CHROM"].astype(int)
    return L.sort_values(["CHROM_N", "POS"]).reset_index(drop=True)

def maf_range(stem):
    p = f"{stem}.acount"
    if not os.path.isfile(p):
        return None
    a = pd.read_csv(p, sep=r"\s+")
    alt = a["ALT_CTS"] if "ALT_CTS" in a.columns else a["ALT_CT"]
    maf = np.minimum(alt, a["OBS_CT"] - alt) / a["OBS_CT"]
    return maf.min(), maf.max()

E, V, W, LAB = {}, {}, {}, {}
for k, stem in FITS.items():
    E[k] = read_eigenvec(stem)
    ev = np.atleast_1d(np.loadtxt(f"{stem}.eigenval"))
    V[k] = ev / ev.sum() * 100
    W[k] = read_weights(stem)
    mr = maf_range(stem)
    LAB[k] = (f"{k}\nMAF {mr[0]*100:.3f}-{mr[1]*100:.3f}%\n{len(W[k]):,} variants"
              if mr else f"{k}\n{len(W[k]):,} variants")
    print(f"{k:10s} n={len(E[k]):>7,}  variants={len(W[k]):>7,}  "
          + (f"MAF {mr[0]:.5f}-{mr[1]:.5f}" if mr else ""))

## Scree by fraction

If the lowest fraction has a flatter spectrum than the highest, its PCs are
closer to noise - worth knowing before reading anything into its loadings.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for k in FITS:
    ax.plot(range(1, len(V[k]) + 1), V[k], "-o", ms=4, label=k)
ax.set_xlabel("PC"); ax.set_ylabel("% variance explained")
ax.set_xticks(range(1, N_PCS + 1)); ax.legend(fontsize=8)
ax.set_title(f"{ARM} - eigenvalue spectrum by MAF fraction")
plt.tight_layout(); plt.savefig(f"{OUT}/frac_scree.png", dpi=150); plt.show()

## Loadings by fraction, PC1-10

Rows are PCs, columns are fractions. Chromosomes alternate colour. Each row
shares its y-axis across fractions so heights are directly comparable.

In [ ]:
def manhattan(ax, L, col, ymax=None):
    off, ticks, labels = 0.0, [], []
    for c in sorted(L["CHROM_N"].unique()):
        g = L[L["CHROM_N"] == c]
        ax.scatter(off + g["POS"].to_numpy(), g[col].abs(),
                   s=0.35, alpha=0.5, rasterized=True,
                   color="0.25" if c % 2 else "tab:blue")
        ticks.append(off + g["POS"].mean()); labels.append(str(c))
        off += g["POS"].max() + 2e7
    ax.set_xlim(0, off)
    ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=5)
    if ymax:
        ax.set_ylim(0, ymax * 1.05)
    return ticks, labels

keys = list(FITS)
fig, axes = plt.subplots(N_PCS, len(keys),
                         figsize=(4.6 * len(keys), 1.55 * N_PCS), squeeze=False)
for r in range(N_PCS):
    col = f"PC{r+1}"
    ymax = max(W[k][col].abs().max() for k in keys if col in W[k].columns)
    for c, k in enumerate(keys):
        ax = axes[r, c]
        if col not in W[k].columns:
            ax.axis("off"); continue
        manhattan(ax, W[k], col, ymax)
        if c == 0:
            ax.set_ylabel(col, fontsize=9)
        else:
            ax.set_yticklabels([])
        if r == 0:
            ax.set_title(LAB[k], fontsize=8)
        if r < N_PCS - 1:
            ax.set_xticklabels([])
plt.suptitle(f"{ARM} - |allele weight| by genomic position, PC1-{N_PCS}, "
             f"by MAF fraction", y=0.999)
plt.tight_layout()
plt.savefig(f"{OUT}/frac_loadings_pc1_{N_PCS}.png", dpi=140, bbox_inches="tight")
plt.show()

## Loading concentration by fraction

The Manhattan grid read as one number per panel: share of squared loading mass in
the top 1% of variants, **as a ratio to its own null**.

The null matters here and is easy to get wrong. For iid Gaussian loadings the top
1% of variants already carry ~8% of the squared mass, because w^2 is heavy-tailed
- so comparing against "uniform = 1%" would make every PC look pathologically
concentrated. The null below is simulated per panel at that panel's exact variant
count (it drifts with n), and the statistic reported is
`enrichment = observed / null`. Enrichment ~1 means indistinguishable from
unstructured loadings; >1.5 is worth looking at in the Manhattan grid.

In [ ]:
RNG = np.random.default_rng(0)
N_NULL = 40

def top1pct(w2):
    n1 = max(1, len(w2) // 100)
    return np.sort(w2)[-n1:].sum() / w2.sum()

def null_top1pct(nvar, reps=N_NULL):
    """Expected top-1% share for iid Gaussian loadings at this variant count."""
    return float(np.mean([top1pct(RNG.normal(size=nvar) ** 2) for _ in range(reps)]))

nulls = {k: null_top1pct(len(W[k])) for k in keys}
print("simulated null top-1% share (iid Gaussian loadings):")
for k in keys:
    print(f"  {k:10s} n={len(W[k]):>7,}  null={nulls[k]:.4f}")

rows = []
for k in keys:
    for r in range(1, N_PCS + 1):
        col = f"PC{r}"
        if col not in W[k].columns:
            continue
        obs = top1pct(W[k][col].to_numpy() ** 2)
        rows.append({"fraction": k, "PC": r, "top1pct_share": obs,
                     "null_share": nulls[k], "enrichment": obs / nulls[k],
                     "pct_var": V[k][r - 1]})
conc = pd.DataFrame(rows)
piv = conc.pivot(index="PC", columns="fraction", values="enrichment")[keys]
print("\nenrichment over null (1.0 = indistinguishable from unstructured):")
print(piv.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for k in keys:
    d = conc[conc["fraction"] == k]
    axes[0].plot(d["PC"], d["enrichment"], "-o", ms=4, label=k)
axes[0].axhline(1.0, color="crimson", ls="--", lw=1, label="null")
axes[0].set_xlabel("PC"); axes[0].set_ylabel("top 1% share / null")
axes[0].set_xticks(range(1, N_PCS + 1)); axes[0].legend(fontsize=7)
axes[0].set_title("loading concentration, relative to null")

A = piv.to_numpy()
im = axes[1].imshow(A, aspect="auto", cmap="magma", vmin=min(1.0, np.nanmin(A)))
axes[1].set_xticks(range(len(keys)))
axes[1].set_xticklabels(keys, fontsize=7, rotation=30, ha="right")
axes[1].set_yticks(range(N_PCS)); axes[1].set_yticklabels(range(1, N_PCS + 1))
axes[1].set_ylabel("PC"); axes[1].set_title("enrichment over null")
for a in range(A.shape[0]):
    for b in range(A.shape[1]):
        axes[1].text(b, a, f"{A[a, b]:.2f}", ha="center", va="center", fontsize=6,
                     color="white" if A[a, b] < np.nanmax(A) / 2 else "black")
fig.colorbar(im, ax=axes[1], fraction=0.04)
plt.tight_layout(); plt.savefig(f"{OUT}/frac_concentration.png", dpi=150); plt.show()

## PC scatters by fraction, PC1-10

One row per fraction, five pairs covering PC1-10. Hexbin with log counts.

In [ ]:
PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10)]
fig, axes = plt.subplots(len(keys), len(PAIRS),
                         figsize=(3.5 * len(PAIRS), 3.2 * len(keys)), squeeze=False)
for r, k in enumerate(keys):
    d = E[k]
    for c, (i, j) in enumerate(PAIRS):
        ax = axes[r, c]
        if f"PC{i}" not in d.columns or f"PC{j}" not in d.columns:
            ax.axis("off"); continue
        hb = ax.hexbin(d[f"PC{i}"], d[f"PC{j}"], gridsize=95, norm=LogNorm(),
                       cmap="viridis", mincnt=1)
        ax.set_xlabel(f"PC{i}", fontsize=8); ax.set_ylabel(f"PC{j}", fontsize=8)
        if c == 0:
            ax.text(-0.32, 0.5, LAB[k], transform=ax.transAxes, fontsize=8,
                    rotation=90, va="center", ha="center")
        if c == len(PAIRS) - 1:
            fig.colorbar(hb, ax=ax, fraction=0.04, label="individuals")
plt.suptitle(f"{ARM} - PC scatters by MAF fraction (n per row varies)", y=0.999)
plt.tight_layout()
plt.savefig(f"{OUT}/frac_pcpairs_pc1_{N_PCS}.png", dpi=140, bbox_inches="tight")
plt.show()

## Do the fractions recover the same axes?

|r| between each fraction's PCs and the full arm's, over shared individuals. PCs
are identified up to sign and up to rotation within near-degenerate subspaces, so
`max |r|` across all pairs is the honest statistic; the diagonal is reported
alongside it.

In [ ]:
ref = "full arm" if "full arm" in E else keys[0]
rows = []
mats = {}
for k in keys:
    if k == ref:
        continue
    a, b = E[ref], E[k]
    ids = a.index.intersection(b.index)
    A, B = a.loc[ids].to_numpy(), b.loc[ids].to_numpy()
    R = np.abs(np.corrcoef(A.T, B.T)[:A.shape[1], A.shape[1]:])
    mats[k] = R
    for i in range(R.shape[0]):
        rows.append({"fraction": k, "PC": i + 1,
                     "diag_abs_r": R[i, i] if i < R.shape[1] else np.nan,
                     "max_abs_r": R[i].max(),
                     "best_match": int(R[i].argmax()) + 1, "n": len(ids)})
xf = pd.DataFrame(rows)
if len(xf):
    print(f"each fraction vs '{ref}':")
    print(xf.pivot(index="PC", columns="fraction", values="max_abs_r")
          .round(3).to_string())

    fig, axes = plt.subplots(1, len(mats), figsize=(3.4 * len(mats), 3.4),
                             squeeze=False)
    for ax, (k, R) in zip(axes[0], mats.items()):
        im = ax.imshow(R, cmap="magma", vmin=0, vmax=1)
        ax.set_xticks(range(R.shape[1])); ax.set_xticklabels(range(1, R.shape[1] + 1), fontsize=6)
        ax.set_yticks(range(R.shape[0])); ax.set_yticklabels(range(1, R.shape[0] + 1), fontsize=6)
        ax.set_xlabel(k, fontsize=8); ax.set_ylabel(ref, fontsize=8)
    fig.colorbar(im, ax=axes, fraction=0.02, label="|r|")
    plt.suptitle(f"PC correspondence vs '{ref}'")
    plt.savefig(f"{OUT}/frac_cross_corr.png", dpi=150, bbox_inches="tight")
    plt.show()

## Save and upload

In [ ]:
CONC_TSV = f"{OUT}/frac_concentration.tsv"
XF_TSV   = f"{OUT}/frac_cross_corr.tsv"
conc.to_csv(CONC_TSV, sep="\t", index=False, float_format="%.6f")
xf.to_csv(XF_TSV, sep="\t", index=False, float_format="%.5f")
print(f"wrote {CONC_TSV}\nwrote {XF_TSV}")

subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{CONC_TSV}" "{XF_TSV}" "{OUT_GS}/"
gcloud storage cp "{OUT}/frac_"*.png "{OUT_GS}/figures/"
"""], check=True)
print(f"uploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/rare_pca_plots.ipynb')
_gs = f'{OUT_GS}/notebooks/rare_pca_plots.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')